# Phase 5 runner (thin) — real-data prerequisite gate

Runs **existing repository scripts** on a Kaggle T4. Setup/rules: `docs/PHASE_5_REAL_DATA_GATE.md`.

Before running: **Accelerator = GPU T4**, **Internet = On**, attach (1) the private repo dataset built by
`scripts/package_for_kaggle.py` and (2) a **CheXpert-small dataset you are authorized to use** (it needs `train.csv` + the images).
Use **Save Version -> Save & Run All (Commit)**.

Session 1 (default flags): preflight + ONE PRISM calibration. Session 2 (only if the baseline is judged viable): set `RUN_CLASSIFIER = True`.
Correction session (only after gate 5a on session 1 was judged PARTIAL): `RUN_PREFLIGHT = RUN_CALIBRATION = False`, `RUN_CORRECTION = True`.
Gate 5b session: `RUN_PREFLIGHT = RUN_CALIBRATION = RUN_CORRECTION = False`, `RUN_CLASSIFIER = True` (data validation runs first and blocks training if it fails).
To resume an interrupted classifier run, attach the previous version's *output* as an input.

In [ ]:
import glob, os, shutil, subprocess, sys, time

# ---------------- parameters (edit only this cell) ----------------
RUN_PREFLIGHT = True      # locate CheXpert + 2k-image / 1-epoch classifier pipeline check (a pipeline check, NOT a metric)
RUN_CALIBRATION = True    # ONE real-checkpoint PRISM calibration on ONE real CheXpert image (3 edit passes share 1 inversion)
RUN_CLASSIFIER = False    # full Lean classifier train + test evaluation: set True ONLY after the calibration is judged viable
RUN_CORRECTION = False    # the ONE pre-registered correction attempt (docs/PHASE_5_REAL_DATA_GATE.md section 7); set preflight/calibration False
MIN_VRAM_GB = 8
PRISM_CKPT = "/kaggle/temp/prism/model.safetensors"   # re-downloadable (3.4 GB, SHA-256 verified); not persisted on purpose
RESTORE_FROM = sorted(p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + "experiments"))   # previous output, if attached
# ------------------------------------------------------------------

WORK, REPO = "/kaggle/working", "/kaggle/working/repo"
OUT = f"{WORK}/experiments"
P5 = f"{OUT}/phase5"
os.makedirs(P5, exist_ok=True)
os.environ.update(HF_HOME="/kaggle/temp/hf", PYTHONUNBUFFERED="1")
FAILED = []
LOG_PATH = f"{OUT}/notebook_log.txt"
LOG = open(LOG_PATH, "a")   # persisted copy of everything printed below

def say(text=""):
    print(text, flush=True)
    LOG.write(text + "\n")
    LOG.flush()

def run(name, cmd):
    # stream through Python so output reaches the cell log (and the log file) however the kernel handles subprocess fds
    say(); say(f"=== {name}"); say("$ " + " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=REPO if os.path.isdir(REPO) else None, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        say(line.rstrip("\n"))
    rc = proc.wait()
    say(f"--- exit code {rc}")
    if rc:
        FAILED.append((name, rc))
    return rc

In [ ]:
def find(suffix):
    # shallowest first; explicit depths (not a recursive walk) so large attached datasets are never scanned
    return sorted((p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + suffix)), key=len)

boots = find("scripts/kaggle_bootstrap.py")
if not boots:
    zips = find("*repo*.zip")   # package attached but not auto-extracted by Kaggle
    if zips:
        shutil.unpack_archive(zips[0], "/kaggle/temp/repo_src")
        boots = ["/kaggle/temp/repo_src/scripts/kaggle_bootstrap.py"]
if not boots:
    tree = [f"{r}  {f[:5]}" for r, _, f in os.walk("/kaggle/input") if r.count("/") <= 5]
    say("repo package not found; /kaggle/input tree:"); [say(t) for t in tree[:60]]
    raise AssertionError("repo package not found under /kaggle/input (tree printed above): attach the dataset built by scripts/package_for_kaggle.py")
say(f"using repo package: {boots[0]}")
run("bootstrap (copy repo, install deps, torch guarded)", [sys.executable, boots[0], "--repo-src", os.path.dirname(os.path.dirname(boots[0])), "--workdir", REPO])

In [ ]:
say(f"notebook flags: PREFLIGHT={RUN_PREFLIGHT} CALIBRATION={RUN_CALIBRATION} CORRECTION={RUN_CORRECTION} CLASSIFIER={RUN_CLASSIFIER}")
if not (RUN_PREFLIGHT or RUN_CALIBRATION or RUN_CORRECTION or RUN_CLASSIFIER):
    say("WARNING: every stage flag is False - nothing but the environment check will run")
run("environment check",[sys.executable, "scripts/check_environment.py", "--require-gpu", "--min-vram-gb", str(MIN_VRAM_GB), "--out", f"{OUT}/env_report.json"])
LOCATED = False
if RUN_PREFLIGHT or RUN_CALIBRATION or RUN_CLASSIFIER or RUN_CORRECTION:
    LOCATED = run("locate CheXpert + write configs", [sys.executable, "scripts/locate_chexpert.py", "--out-dir", P5, "--prism-checkpoint", PRISM_CKPT]) == 0

In [ ]:
if RUN_PREFLIGHT and LOCATED:
    run("classifier PREFLIGHT (2k images, 1 epoch - pipeline check only)", [sys.executable, "scripts/run_experiment.py", "--config", f"{P5}/generated_classifier_preflight.yaml", "--name", "phase5_classifier_preflight", "--run-root", OUT])

In [ ]:
if RUN_CALIBRATION and LOCATED:
    if not os.path.exists(PRISM_CKPT):
        run("fetch PRISM checkpoint (3.4 GB, size + SHA-256 verified)", [sys.executable, "scripts/download_prism_checkpoint.py", "--dest", os.path.dirname(PRISM_CKPT)])
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", f"{P5}/generated_calibration.yaml", "--run-root", OUT]
    if run("PRISM calibration (512 px)", cmd) and "OutOfMemoryError" in open(LOG_PATH).read()[-30000:]:
        run("PRISM calibration FALLBACK (384 px; CUDA out of memory at 512 px)", cmd + ["--set", "editing.image_size=384"])

In [ ]:
if RUN_CORRECTION and LOCATED:
    import json
    # same image, weaker self-attention lock (the lever the first run showed to be monotonic); everything else inherited from the calibration config
    img = json.load(open(f"{P5}/chexpert_layout.json"))["calibration_image"]
    inputs = [{"image": img, "pathology": "Cardiomegaly", "operation": "add", "params": {"self_replace_steps": v}} for v in (0.2, 0.0)]
    json.dump({"extends": f"{P5}/generated_calibration.yaml", "experiment": {"name": "phase5_prism_correction"}, "inputs": inputs}, open(f"{P5}/generated_correction.yaml", "w"))
    if not os.path.exists(PRISM_CKPT):
        run("fetch PRISM checkpoint (3.4 GB, size + SHA-256 verified)", [sys.executable, "scripts/download_prism_checkpoint.py", "--dest", os.path.dirname(PRISM_CKPT)])
    run("PRISM correction attempt (512 px, self_replace_steps 0.2 / 0.0)", [sys.executable, "scripts/run_experiment.py", "--config", f"{P5}/generated_correction.yaml", "--run-root", OUT])

In [ ]:
if RUN_CLASSIFIER and LOCATED:
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", f"{P5}/generated_classifier.yaml", "--name", "phase5_classifier", "--run-root", OUT]
    if RESTORE_FROM:
        cmd += ["--restore-from", *RESTORE_FROM]
    # training starts only if the data validation passed (frontal filter, patient-disjoint split, `ignore` policy, images decode)
    valid = run("DATA VALIDATION (must pass before training)", [sys.executable, "scripts/validate_chexpert_data.py", "--config", f"{P5}/generated_classifier.yaml", "--out", f"{P5}/data_validation.json"]) == 0
    if valid and run("classifier train (Lean: 100k frontal images, 4 epochs)", cmd) == 0:
        best = sorted(glob.glob(f"{OUT}/phase5_classifier/*/checkpoints/best.pt"))[-1]
        metrics = os.path.join(os.path.dirname(os.path.dirname(best)), "test_metrics_best.json")
        run("classifier TEST evaluation (best.pt, held-out patients)", [sys.executable, "scripts/evaluate_classifier.py", "--config", f"{P5}/generated_classifier.yaml", "--checkpoint", best, "--device", "cuda", "--split", "test", "--out", metrics])

In [ ]:
# Always archive, even if a step failed, so nothing is lost; fail loudly only afterwards.
archive = shutil.make_archive(f"{WORK}/artifacts_{time.strftime('%Y%m%d-%H%M%S')}", "zip", OUT)
say(f"artifacts: {archive} {os.path.getsize(archive) / 1e6:.2f} MB")
import hashlib
say(f"artifacts sha256: {hashlib.file_digest(open(archive, 'rb'), 'sha256').hexdigest()}   (verify the downloaded zip against this line)")
say(f"failed steps: {FAILED or 'none'}")
if FAILED:
    raise RuntimeError(f"failed steps: {FAILED}")